# 13. Experiment 1 — Corpus statistics

In [12]:
import gzip
import json
import nltk
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.util import ngrams
from tqdm import tqdm

# Tải data của NLTK (chỉ cần chạy lần đầu)
nltk.download('punkt')
nltk.download('punkt_tab')

def process_c4_corpus(filepath, max_docs=None):
    num_docs = 0
    num_sentences = 0
    num_tokens = 0
    
    unigram_counter = Counter()
    bigram_counter = Counter()
    trigram_counter = Counter()
    
    print(f"Đang đọc và xử lý dữ liệu từ: {filepath}")
    
    # Mở file .json.gz
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, total=30000, desc="Processing Documents"):
            if max_docs and num_docs >= max_docs:
                break
                
            # Đọc JSON object từ dòng hiện tại
            data = json.loads(line)
            text = data.get("text", "")
            if not text:
                continue
                
            num_docs += 1
            
            # Tách câu
            sentences = sent_tokenize(text)
            num_sentences += len(sentences)
            
            # Tách từ và tạo n-grams
            for sent in sentences:
                tokens = [word.lower() for word in word_tokenize(sent) if word.isalnum()]
                num_tokens += len(tokens)
                
                unigram_counter.update(tokens)
                bigram_counter.update(ngrams(tokens, 2))
                trigram_counter.update(ngrams(tokens, 3))

    # Đếm số lượng n-gram chỉ xuất hiện 1 lần
    once_unigrams = sum(1 for count in unigram_counter.values() if count == 1)
    once_bigrams = sum(1 for count in bigram_counter.values() if count == 1)
    once_trigrams = sum(1 for count in trigram_counter.values() if count == 1)
    
    stats = {
        "num_docs": num_docs,
        "num_sentences": num_sentences,
        "num_tokens": num_tokens,
        "vocab_size": len(unigram_counter),
        "unique_unigrams": len(unigram_counter),
        "unique_bigrams": len(bigram_counter),
        "unique_trigrams": len(trigram_counter),
        "once_unigrams": once_unigrams,
        "once_bigrams": once_bigrams,
        "once_trigrams": once_trigrams,
    }
    
    return stats, unigram_counter, bigram_counter, trigram_counter

def plot_freq_distribution(counter, title, filename):
    """Vẽ biểu đồ phân bố tần suất trên thang log-log"""
    counts = sorted(counter.values(), reverse=True)
    
    plt.figure(figsize=(10, 6))
    plt.plot(counts, color='b')
    plt.xscale('log')
    plt.yscale('log')
    plt.title(title)
    plt.xlabel('Rank (log scale)')
    plt.ylabel('Frequency (log scale)')
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"Đã lưu biểu đồ: {filename}")

# --- KHỞI CHẠY ---
file_path = r"C:\Users\tasdus_117\Desktop\NLP\lab02\c4-train.00000-of-01024-30K.json.gz"

# Chạy thống kê và xử lý n-grams
stats, unigrams, bigrams, trigrams = process_c4_corpus(file_path)

# In kết quả dạng bảng
print("\n--- CORPUS STATISTICS ---")
print(f"Số documents: {stats['num_docs']:,}")
print(f"Số sentences: {stats['num_sentences']:,}")
print(f"Số tokens:    {stats['num_tokens']:,}")
print(f"Vocabulary:   {stats['vocab_size']:,}\n")

print(f"{'Model':<10} | {'# unique n-grams':<18} | {'# n-grams xuất hiện 1 lần':<25}")
print("-" * 59)
print(f"{'Unigram':<10} | {stats['unique_unigrams']:<18,} | {stats['once_unigrams']:<25,}")
print(f"{'Bigram':<10} | {stats['unique_bigrams']:<18,} | {stats['once_bigrams']:<25,}")
print(f"{'Trigram':<10} | {stats['unique_trigrams']:<18,} | {stats['once_trigrams']:<25,}")

# Vẽ và lưu biểu đồ vào cùng thư mục chạy script
plot_freq_distribution(unigrams, 'Unigram Frequency Distribution (C4 Dataset)', 'unigram_freq.png')
plot_freq_distribution(bigrams, 'Bigram Frequency Distribution (C4 Dataset)', 'bigram_freq.png')
plot_freq_distribution(trigrams, 'Trigram Frequency Distribution (C4 Dataset)', 'trigram_freq.png')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tasdus_117\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tasdus_117\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Đang đọc và xử lý dữ liệu từ: C:\Users\tasdus_117\Desktop\NLP\lab02\c4-train.00000-of-01024-30K.json.gz


Processing Documents: 100%|██████████| 30000/30000 [00:36<00:00, 825.57it/s]



--- CORPUS STATISTICS ---
Số documents: 30,000
Số sentences: 622,194
Số tokens:    10,692,174
Vocabulary:   180,774

Model      | # unique n-grams   | # n-grams xuất hiện 1 lần
-----------------------------------------------------------
Unigram    | 180,774            | 86,013                   
Bigram     | 2,640,066          | 1,886,578                
Trigram    | 6,225,752          | 5,371,879                
Đã lưu biểu đồ: unigram_freq.png
Đã lưu biểu đồ: bigram_freq.png
Đã lưu biểu đồ: trigram_freq.png


# 16. Experiment 2 — MLE vs smoothing

In [19]:
import math
from nltk.tokenize import word_tokenize
from collections import defaultdict, Counter

class NGramLanguageModel:
    def __init__(self, n, k_smoothing=1.0, unk_threshold=1):
        self.n = n
        self.k = k_smoothing
        self.unk_threshold = unk_threshold
        self.UNK = "<UNK>"
        self.START = "<s>"
        self.END = "</s>"
        
        self.vocab = set()
        self.vocab_size = 0
        self.ngram_counts = defaultdict(int)
        self.context_counts = defaultdict(int)

    def _pad_sequence(self, tokens):
        pad_length = max(1, self.n - 1)
        return [self.START] * pad_length + tokens + [self.END]

    def build_vocabulary(self, corpus_sentences):
        word_freq = Counter()
        for sentence in corpus_sentences:
            word_freq.update(word_tokenize(sentence.lower()))
            
        self.vocab = {word for word, count in word_freq.items() if count >= self.unk_threshold}
        self.vocab.update([self.UNK, self.END])
        self.vocab_size = len(self.vocab)

    def _replace_oov(self, tokens):
        return [word if word in self.vocab else self.UNK for word in tokens]

    def fit(self, corpus_sentences):
        self.build_vocabulary(corpus_sentences)
        
        for sentence in corpus_sentences:
            tokens = self._replace_oov(word_tokenize(sentence.lower()))
            padded_tokens = self._pad_sequence(tokens)
            
            for i in range(self.n - 1, len(padded_tokens)):
                context = tuple(padded_tokens[i - self.n + 1 : i])
                word = padded_tokens[i]
                
                self.ngram_counts[context + (word,)] += 1
                self.context_counts[context] += 1
    def probability(self, context, word):
        """Tính xác suất P(word | context) áp dụng Add-k smoothing"""
        context = tuple(context)
        ngram = context + (word,)
        
        # P(w | c) = (C(c, w) + k) / (C(c) + k * |V|)
        numerator = self.ngram_counts[ngram] + self.k
        denominator = self.context_counts[context] + (self.k * self.vocab_size)
        
        # Xử lý tránh chia cho 0 nếu k=0 (MLE)
        if denominator == 0:
            return 0.0
            
        return numerator / denominator

    def next_word_distribution(self, context):
        """
        Trả về phân bố xác suất cho tất cả các từ trong vocab cho từ tiếp theo,
        dựa trên ngữ cảnh đưa vào.
        """
        context = tuple(context)
        dist = {}
        for word in self.vocab:
            # Bỏ qua các token không bao giờ được dự đoán là từ tiếp theo
            if word in [self.START, self.UNK]: 
                continue
            
            p = self.probability(context, word)
            if p > 0:
                dist[word] = p
            
        # Sắp xếp giảm dần theo xác suất
        dist = dict(sorted(dist.items(), key=lambda item: item[1], reverse=True))
        return dist
# ==========================================
# THỰC NGHIỆM 2: MLE vs LAPLACE SMOOTHING
# ==========================================

def evaluate_perplexity(model, corpus_sentences):
    """
    Tính PPL. Bắt lỗi xác suất = 0 đối với MLE để trả về vô cực (inf).
    """
    total_log_prob = 0.0
    total_predicted_tokens = 0
    
    for sentence in corpus_sentences:
        tokens = model._replace_oov(word_tokenize(sentence.lower()))
        padded_tokens = model._pad_sequence(tokens)
        
        sentence_log_prob = 0.0
        
        for i in range(model.n - 1, len(padded_tokens)):
            context = tuple(padded_tokens[i - model.n + 1 : i])
            word = padded_tokens[i]
            
            # C(context, word) + k
            numerator = model.ngram_counts[context + (word,)] + model.k
            # C(context) + k*|V|
            denominator = model.context_counts[context] + (model.k * model.vocab_size)
            
            # MLE (k=0) gặp n-gram chưa từng thấy -> Xác suất = 0
            if numerator == 0 or denominator == 0:
                sentence_log_prob = float('-inf')
                break
                
            prob = numerator / denominator
            sentence_log_prob += math.log(prob)
            
        total_log_prob += sentence_log_prob
        
        # Số lượng từ cần dự đoán trong câu = len(padded) - độ dài context
        total_predicted_tokens += len(padded_tokens) - (model.n - 1)
        
    # Nếu có bất kỳ câu nào có xác suất = 0 (log_prob = -inf), PPL của cả tập là Vô cực
    if total_log_prob == float('-inf'):
        return float('inf')
        
    # Công thức PPL = exp( - (1/N) * sum(log P) )
    avg_log_prob = total_log_prob / total_predicted_tokens
    return math.exp(-avg_log_prob)

def run_experiment_2(train_set, valid_set, test_set):
    # Khởi tạo 4 mô hình: MLE (k=0) và Laplace (k=1) cho Bigram (n=2) và Trigram (n=3)
    models = {
        "Bigram MLE": NGramLanguageModel(n=2, k_smoothing=0.0),
        "Bigram Laplace": NGramLanguageModel(n=2, k_smoothing=1.0),
        "Trigram MLE": NGramLanguageModel(n=3, k_smoothing=0.0),
        "Trigram Laplace": NGramLanguageModel(n=3, k_smoothing=1.0),
    }

    results = []
    
    # Huấn luyện và đánh giá
    for name, model in models.items():
        model.fit(train_set)
        
        train_ppl = evaluate_perplexity(model, train_set)
        valid_ppl = evaluate_perplexity(model, valid_set)
        test_ppl  = evaluate_perplexity(model, test_set)
        
        results.append((name, train_ppl, valid_ppl, test_ppl))

    # In kết quả dạng bảng
    print(f"{'Model':<18} | {'Train PPL':<12} | {'Valid PPL':<12} | {'Test PPL':<12}")
    print("-" * 63)
    for name, train_p, valid_p, test_p in results:
        # Định dạng float hiển thị 2 chữ số thập phân, vô cực in chữ "inf"
        t_str = f"{train_p:.2f}" if train_p != float('inf') else "inf"
        v_str = f"{valid_p:.2f}" if valid_p != float('inf') else "inf"
        te_str = f"{test_p:.2f}" if test_p != float('inf') else "inf"
        
        print(f"{name:<18} | {t_str:<12} | {v_str:<12} | {te_str:<12}")

In [16]:
import gzip
import json
import random
from tqdm import tqdm
from nltk.tokenize import sent_tokenize

def load_and_split_c4(filepath, max_docs=30000):
    docs = []
    print(f"Đang đọc dữ liệu từ: {filepath}")
    
    # Đọc file nén
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(tqdm(f, desc="Reading Docs", total=max_docs)):
            if max_docs and i >= max_docs:
                break
            data = json.loads(line)
            text = data.get("text", "")
            if text:
                docs.append(text)
                
    # Xáo trộn dữ liệu ngẫu nhiên với seed cố định (để mỗi lần chạy kết quả chia không bị đổi)
    random.seed(42)
    random.shuffle(docs)
    
    # Tính số lượng cho tỷ lệ 80 - 10 - 10
    total_docs = len(docs)
    train_size = int(0.8 * total_docs)
    valid_size = int(0.1 * total_docs)
    
    train_docs = docs[:train_size]
    valid_docs = docs[train_size : train_size + valid_size]
    test_docs  = docs[train_size + valid_size :]
    
    print(f"\nĐã chia Documents: Train={len(train_docs):,}, Valid={len(valid_docs):,}, Test={len(test_docs):,}")
    
    # Hàm phụ để tách list các văn bản thành list các câu
    def get_sentences(doc_list, desc):
        sentences = []
        for doc in tqdm(doc_list, desc=desc):
            sentences.extend(sent_tokenize(doc))
        return sentences
        
    print("\nĐang tách câu (Sentence Tokenization)...")
    train_sentences = get_sentences(train_docs, "Train Sentences")
    valid_sentences = get_sentences(valid_docs, "Valid Sentences")
    test_sentences  = get_sentences(test_docs, "Test Sentences")
    
    return train_sentences, valid_sentences, test_sentences

# ==========================================
# GỌI HÀM THỰC THI CHÍNH
# ==========================================
if __name__ == "__main__":
    file_path = r"C:\Users\tasdus_117\Desktop\NLP\lab02\c4-train.00000-of-01024-30K.json.gz"
    
    DOCS_TO_LOAD = 30000  # Thay thành 30000 khi chạy thật
    
    train_set, valid_set, test_set = load_and_split_c4(file_path, max_docs=DOCS_TO_LOAD)
    
    print(f"\nTổng số câu: Train={len(train_set):,}, Valid={len(valid_set):,}, Test={len(test_set):,}")
    print("\n" + "="*50)
    print("Bắt đầu chạy Experiment 2: MLE vs Laplace Smoothing")
    print("="*50 + "\n")
    
    # Gọi hàm run_experiment_2 từ code block trước
    run_experiment_2(train_set, valid_set, test_set)

Đang đọc dữ liệu từ: C:\Users\tasdus_117\Desktop\NLP\lab02\c4-train.00000-of-01024-30K.json.gz


Reading Docs: 100%|██████████| 30000/30000 [00:00<00:00, 73529.67it/s]



Đã chia Documents: Train=24,000, Valid=3,000, Test=3,000

Đang tách câu (Sentence Tokenization)...


Test Sentences: 100%|██████████| 3000/3000 [00:00<00:00, 7914.13it/s]



Tổng số câu: Train=500,677, Valid=60,734, Test=60,783

Bắt đầu chạy Experiment 2: MLE vs Laplace Smoothing

Model              | Train PPL    | Valid PPL    | Test PPL    
---------------------------------------------------------------
Bigram MLE         | 125.98       | inf          | inf         
Bigram Laplace     | 4084.20      | 5182.81      | 5174.02     
Trigram MLE        | 13.37        | inf          | inf         
Trigram Laplace    | 32799.17     | 53005.42     | 52914.29    


1. Tại sao Valid/Test PPL của MLE lại luôn bằng vô cực?
Nguyên nhân (Zero-Frequency Problem): Mô hình MLE dựa trên những gì nó đã thấy ở tập Train. Trên tập Valid và Test (chứa tới hơn 60,000 câu), chắc chắn sẽ xuất hiện rất nhiều cặp từ hoặc cụm 3 từ mới chưa từng có trong tập Train.
Hệ quả toán học: MLE gán xác suất bằng $0$ cho các n-gram mới này. Vì xác suất của cả câu là phép nhân, chỉ cần một n-gram có xác suất $0$, cả câu sẽ có xác suất $0$. Khi đưa vào công thức Perplexity, PPL lập tức bị đẩy lên vô cực ($\infty$).

2. Tại sao trên tập Train, Trigram MLE (13.37) lại tốt hơn hẳn Bigram MLE (125.98)?
Trigram có ngữ cảnh dài hơn. Nhờ vậy, tính cụ thể của dự đoán cao hơn rất nhiều.
Trên tập Train, mô hình đã nhớ toàn bộ các cụm từ này. Ngữ cảnh càng dài thì mô hình càng Overfit tốt, dẫn đến độ tự tin cực kỳ cao và PPL giảm xuống mức rất thấp (13.37).
3. Tại sao khi dùng Laplace, Train PPL lại tăng vọt so với MLE?
Nhìn vào bảng: Bigram Train PPL tăng từ 125.98 lên 4084.20.
Cơ chế chiết khấu: Kỹ thuật Laplace hoạt động bằng cách cộng thêm 1 vào tử số và cộng thêm $\vert{}V\vert{}$ vào mẫu số.
Bản chất của việc này là mô hình phải ăn bớt khối lượng xác suất của các n-gram đã từng xuất hiện để chia đều cho vô vàn các n-gram chưa từng xuất hiện. Do đó, mức độ tự tin của mô hình trên chính tập Train bị giảm sút nghiêm trọng, làm PPL trên tập Train tăng vọt. Đổi lại, mô hình không bị chết khi gặp từ mới trên tập Valid/Test.
4. Vấn đề thảm họa: Tại sao Trigram Laplace (~53,000) lại tệ hơn Bigram Laplace (~5,100)?
Đây là luận điểm quan trọng nhất của bài Lab, thể hiện rõ giới hạn của Laplace Smoothing khi đối mặt với Data Sparsity.
Tập từ vựng $\vert{}V\vert{}$ của bạn rất lớn. Không gian của Bigram là $\vert{}V\vert{}^2$, nhưng không gian của Trigram là $\vert{}V\vert{}^3$. Khối lượng cụm 3 từ có thể tồn tại là vô cùng khổng lồ, nhưng hầu hết chúng không bao giờ xuất hiện trong thực tế (ví dụ: "con chó đang", "con chó bay", "con chó bàn"...).
Khi áp dụng Laplace cho Trigram, mẫu số bị cộng thêm một lượng khổng lồ. Khối lượng xác suất dự trữ bị chia vụn ra cho hàng chục triệu Trigram "rác".
Hệ quả là, những Trigram có thật (xuất hiện trong tập Valid/Test) bị gán cho một mức xác suất cực kỳ nhỏ bé và gần như bằng nhau (uniform). Sự phân phối quá cào bằng này làm mô hình mất đi khả năng dự đoán chính xác, đẩy Perplexity lên một con số khổng lồ (hơn 53,000).
Kết luận cho báo cáo: Laplace Smoothing là một phương pháp quá ngây thơ. Nó hoạt động tạm ổn cho Unigram hoặc Bigram, nhưng bị sụp đổ hoàn toàn trên Trigram do không gian dữ liệu quá thưa thớt. Đó là lý do trong thực tế, các mô hình ngôn ngữ truyền thống phải sử dụng các kỹ thuật làm mịn tiên tiến hơn như Kneser-Ney Smoothing hoặc lùi bậc từ Trigram xuống Bigram nếu không tìm thấy.

# 19. Experiment 3 — Perplexity

In [17]:
import math
from nltk.tokenize import word_tokenize

def evaluate_perplexity(model, corpus_sentences):
    """
    Hàm tính Perplexity cho mô hình N-gram.
    """
    total_log_prob = 0.0
    total_predicted_tokens = 0
    
    for sentence in corpus_sentences:
        tokens = model._replace_oov(word_tokenize(sentence.lower()))
        padded_tokens = model._pad_sequence(tokens)
        
        sentence_log_prob = 0.0
        
        # Bắt đầu duyệt từ vị trí max(1, n-1) để bỏ qua việc dự đoán token <s> ở đầu câu
        start_idx = max(1, model.n - 1)
        
        for i in range(start_idx, len(padded_tokens)):
            context = tuple(padded_tokens[i - model.n + 1 : i])
            word = padded_tokens[i]
            
            numerator = model.ngram_counts[context + (word,)] + model.k
            denominator = model.context_counts[context] + (model.k * model.vocab_size)
            
            # Xử lý an toàn nếu k=0 (MLE)
            if numerator == 0 or denominator == 0:
                sentence_log_prob = float('-inf')
                break
                
            prob = numerator / denominator
            sentence_log_prob += math.log(prob)
            
        total_log_prob += sentence_log_prob
        total_predicted_tokens += len(padded_tokens) - start_idx
        
    if total_log_prob == float('-inf'):
        return float('inf')
        
    avg_log_prob = total_log_prob / total_predicted_tokens
    return math.exp(-avg_log_prob)

def run_experiment_3(train_set, valid_set, test_set):
    """
    Khởi tạo và so sánh Unigram, Bigram, Trigram (sử dụng Laplace Smoothing k=1)
    """
    print("\n" + "="*60)
    print("THỰC NGHIỆM: SO SÁNH UNIGRAM, BIGRAM, TRIGRAM (LAPLACE)")
    print("="*60 + "\n")
    
    # 1. Khởi tạo mô hình
    models = {
        "Unigram": NGramLanguageModel(n=1, k_smoothing=1.0),
        "Bigram":  NGramLanguageModel(n=2, k_smoothing=1.0),
        "Trigram": NGramLanguageModel(n=3, k_smoothing=1.0)
    }
    
    results = []
    
    # 2. Huấn luyện và Đánh giá
    for name, model in models.items():
        print(f"Đang huấn luyện {name} model...")
        model.fit(train_set)
        
        print(f"Đang tính PPL cho {name}...")
        train_ppl = evaluate_perplexity(model, train_set)
        valid_ppl = evaluate_perplexity(model, valid_set)
        test_ppl  = evaluate_perplexity(model, test_set)
        
        results.append((name, train_ppl, valid_ppl, test_ppl))
        
    # 3. In kết quả dạng bảng Markdown chuẩn
    print("\nKẾT QUẢ:\n")
    print(f"{'Model':<12} | {'Train PPL':<15} | {'Validation PPL':<15} | {'Test PPL':<15}")
    print("-" * 65)
    for name, train_p, valid_p, test_p in results:
        t_str = f"{train_p:.2f}" if train_p != float('inf') else "inf"
        v_str = f"{valid_p:.2f}" if valid_p != float('inf') else "inf"
        te_str = f"{test_p:.2f}" if test_p != float('inf') else "inf"
        print(f"{name:<12} | {t_str:<15} | {v_str:<15} | {te_str:<15}")


# ==========================================
# GỌI HÀM THỰC THI CHÍNH
# ==========================================
if __name__ == "__main__":
    run_experiment_3(train_set, valid_set, test_set)


THỰC NGHIỆM: SO SÁNH UNIGRAM, BIGRAM, TRIGRAM (LAPLACE)

Đang huấn luyện Unigram model...
Đang tính PPL cho Unigram...
Đang huấn luyện Bigram model...
Đang tính PPL cho Bigram...
Đang huấn luyện Trigram model...
Đang tính PPL cho Trigram...

KẾT QUẢ:

Model        | Train PPL       | Validation PPL  | Test PPL       
-----------------------------------------------------------------
Unigram      | 1401.36         | 1478.27         | 1470.08        
Bigram       | 4084.20         | 5182.81         | 5174.02        
Trigram      | 32799.17        | 53005.42        | 52914.29       


1. Tại sao Training Perplexity thay đổi?
Hiện tượng: Train PPL tăng mạnh khi tăng bậc N.
Giải thích: Điều này là do tác dụng phụ của phương pháp làm mịn Laplace. Công thức tính xác suất là: $P = \frac{Count + 1}{Total\_Count + \vert{}V\vert{}}$.
Ở Unigram, mẫu số $Total\_Count$ rất lớn, nên việc cộng thêm $\vert{}V\vert{}$ không làm biến dạng xác suất quá nhiều.
Ở Bigram và Trigram, mẫu số là số lần xuất hiện của ngữ cảnh. Đối với cụm 2 từ hoặc 3 từ, $C(context)$ thường rất nhỏ. Khi cộng thêm tập từ vựng $\vert{}V\vert{}$ khổng lồ vào một mẫu số nhỏ, xác suất dự đoán đúng các từ trên chính tập Train bị pha loãng và cào bằng mạnh mẽ. Mô hình đánh mất độ tự tin, khiến độ bối rối trên tập Train tăng vọt theo cấp số nhân khi bậc N tăng.
2. Tại sao Validation/Test có thể không cải thiện?
Hiện tượng: Test PPL không những không cải thiện mà còn tệ đi rất nhiều.
Giải thích: Do vấn đề Data Sparsity bùng nổ ở các không gian chiều cao.
Không gian của Trigram là $\vert{}V\vert{}^3$. Hầu hết các cụm 3 từ trong tập Test là những n-gram hoàn toàn mới chưa từng xuất hiện ở tập Train.
Kỹ thuật Laplace xử lý việc này một cách quá ngây thơ: nó ăn bớt xác suất của các n-gram có thật để chia đều cho vô vàn n-gram rác. Hệ quả là khi gặp một cụm 3 từ có thật trên tập Test, mô hình gán cho nó một xác suất cực thấp. Xác suất quá nhỏ đẩy Perplexity lên con số khổng lồ. Điều này cho thấy Laplace Smoothing không có khả năng xử lý Trigram trong thực tế.
3. Dấu hiệu của Overfitting
Dấu hiệu Overfitting được thể hiện qua khoảng cách giữa Train PPL và Validation/Test PPL:
Unigram: Train PPL và Test PPL rất sát nhau $\rightarrow$ Mô hình đơn giản, không bị Overfitting nhưng mang tính chất Underfitting.
Bigram: Khoảng cách bắt đầu mở rộng $\rightarrow$ Mô hình bắt đầu học thuộc các cặp từ trong Train nhưng bối rối với cặp từ mới ở Test.
Trigram: Khoảng cách cực lớn, chênh lệch hơn 20,000 đơn vị $\rightarrow$ Overfitting nghiêm trọng. Mô hình Trigram học thuộc lòng một số ngữ cảnh dài cụ thể trong tập Train rất tốt, nhưng vì quá phụ thuộc vào các chuỗi dài này, nó hoàn toàn mất khả năng tổng quát hóa khi ngữ cảnh ở tập Valid/Test hơi khác đi một chút.
4. Tác động của Corpus Size
Kết quả này phản ánh trực tiếp giới hạn của kích thước tập dữ liệu:
Kích thước hiện tại: Dù 500,000 câu nghe có vẻ nhiều, nhưng nó vẫn là muối bỏ bể so với không gian $\vert{}V\vert{}^3$ của Trigram. Do corpus không đủ lớn để cung cấp tần suất thống kê đáng tin cậy cho các cụm 3 từ, Laplace smoothing phá hỏng mô hình. Ở kích thước này, Unigram và Bigram hoạt động ổn định hơn.
Nếu Corpus Size tăng lên: Dữ liệu sẽ dày đặc hơn. Không gian Trigram dần được lấp đầy, mẫu số $C(context)$ sẽ đủ lớn để áp đảo nhiễu cộng thêm của thuật toán làm mịn. Tỷ lệ cụm từ mới trên tập Test cũng sẽ giảm đi. Khi đó, sức mạnh của Trigram mới được phát huy, Train PPL và Test PPL sẽ đồng loạt giảm sâu và vượt qua Unigram/Bigram. Tăng corpus size là phương pháp tối ưu nhất để giải quyết Data Sparsity.

# 20. Application — Next-word prediction & 21. Application — Sentence ranking

In [20]:
import math
from nltk.tokenize import word_tokenize

# ==========================================
# APPLICATION 1: NEXT-WORD PREDICTION
# ==========================================
def predict_next_word(model, context_str, top_k=3):
    """
    Dự đoán top-k từ tiếp theo dựa trên ngữ cảnh đầu vào.
    """
    # Tokenize và lấy N-1 từ cuối cùng làm context cho mô hình
    tokens = word_tokenize(context_str.lower())
    n_minus_1 = max(1, model.n - 1)
    
    if len(tokens) >= n_minus_1:
        context = tuple(tokens[-n_minus_1:])
    else:
        # Pad thêm <s> nếu context quá ngắn
        padded = [model.START] * (n_minus_1 - len(tokens)) + tokens
        context = tuple(padded)
        
    dist = model.next_word_distribution(context)
    
    print(f"Input:\n{context_str}")
    print("Output:")
    for i, (word, prob) in enumerate(list(dist.items())[:top_k], 1):
        print(f"{i}. {word:<10} {prob:.4f}")
    print("-" * 30)

def run_app_1_experiments(model):
    print("\n--- EXPERIMENT: NEXT-WORD PREDICTION ---")
    contexts = [
        "the cat",
        "natural language",
        "machine learning",
        "students at",
        "artificial intelligence"
    ]
    
    # Giả lập bảng kết quả (Actual next word tùy thuộc vào dataset của bạn)
    print(f"{'Context':<25} | {'Prediction (Top 1)':<20} | {'Actual next word (Ví dụ)'}")
    print("-" * 75)
    for ctx in contexts:
        tokens = word_tokenize(ctx.lower())
        n_minus_1 = max(1, model.n - 1)
        model_ctx = tuple(tokens[-n_minus_1:]) if len(tokens) >= n_minus_1 else tuple([model.START] * (n_minus_1 - len(tokens)) + tokens)
        
        dist = model.next_word_distribution(model_ctx)
        top_1_word = list(dist.keys())[0] if dist else "<UNK>"
        
        # Actual word tự giả định để minh họa bảng
        actual = "processing" if ctx == "natural language" else "models"
        
        print(f"{ctx:<25} | {top_1_word:<20} | {actual}")


# ==========================================
# APPLICATION 2: SENTENCE RANKING
# ==========================================
def rank_continuations(model, context_str, continuations):
    """
    Tính Log Probability của các phần tiếp nối (continuations) dựa trên context cho trước,
    từ đó xếp hạng độ tự nhiên/hợp lý của chúng.
    """
    context_tokens = word_tokenize(context_str.lower())
    context_tokens = model._replace_oov(context_tokens)
    
    scores = []
    
    for cont in continuations:
        cont_tokens = word_tokenize(cont.lower())
        cont_tokens = model._replace_oov(cont_tokens)
        
        # Ghép context và continuation lại
        padded_full = [model.START] * max(1, model.n - 1) + context_tokens + cont_tokens + [model.END]
        
        log_prob = 0.0
        # Chỉ tính điểm cho phần continuation và token </s> (bỏ qua điểm của phần context)
        start_eval_idx = max(1, model.n - 1) + len(context_tokens)
        
        for i in range(start_eval_idx, len(padded_full)):
            ctx = tuple(padded_full[i - model.n + 1 : i])
            word = padded_full[i]
            
            numerator = model.ngram_counts[ctx + (word,)] + model.k
            denominator = model.context_counts[ctx] + (model.k * model.vocab_size)
            
            if numerator == 0 or denominator == 0:
                log_prob = float('-inf')
                break
                
            p = numerator / denominator
            log_prob += math.log(p)
            
        scores.append((cont, log_prob))
        
    # Xếp hạng giảm dần theo Log Probability
    scores.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\nContext:\n{context_str}")
    print("Ranked Continuations:")
    for i, (cont, score) in enumerate(scores, 1):
        print(f"{i}. {cont:<25} (Log Prob: {score:.4f})")

# --- KHỞI CHẠY TEST 2 ỨNG DỤNG ---
if __name__ == "__main__":
    # Giả định model là một Bigram hoặc Trigram đã được train (ví dụ: trigram_model)
    # Lấy dữ liệu nhỏ gọn để test nhanh chức năng
    test_corpus = [
        "the cat eats fish.",
        "the cat likes milk.",
        "natural language processing is useful for NLP.",
        "machine learning studies language models.",
        "students at vnu-hus are smart.",
        "artificial intelligence is the future."
    ]
    
    app_model = NGramLanguageModel(n=3, k_smoothing=1.0)
    app_model.fit(test_corpus)
    
    # Chạy App 1
    predict_next_word(app_model, "the cat", top_k=3)
    run_app_1_experiments(app_model)
    
    # Chạy App 2
    continuations = [
        "is useful for NLP",
        "banana computer quickly",
        "studies language models"
    ]
    rank_continuations(app_model, "machine learning", continuations)

Input:
the cat
Output:
1. likes      0.0667
2. eats       0.0667
3. smart      0.0333
------------------------------

--- EXPERIMENT: NEXT-WORD PREDICTION ---
Context                   | Prediction (Top 1)   | Actual next word (Ví dụ)
---------------------------------------------------------------------------
the cat                   | likes                | models
natural language          | processing           | processing
machine learning          | studies              | models
students at               | vnu-hus              | models
artificial intelligence   | is                   | models

Context:
machine learning
Ranked Continuations:
1. studies language models   (Log Prob: -11.3897)
2. banana computer quickly   (Log Prob: -13.3639)
3. is useful for NLP         (Log Prob: -15.4151)


# 23. Một thí nghiệm rất quan trọng: context length

### Trả lời câu hỏi: Context dài hơn có luôn tốt hơn không?

Không, context dài hơn không phải lúc nào cũng tốt hơn trong các mô hình ngôn ngữ truyền thống như N-gram. Mặc dù tăng bậc $N$ (từ Unigram $\rightarrow$ Bigram $\rightarrow$ Trigram) giúp mô hình nắm bắt ngữ cảnh tốt hơn trên tập huấn luyện, nhưng nó lại mang đến những hạn chế nghiêm trọng khi đánh giá trên dữ liệu thực tế.

Cụ thể được phân tích qua các yếu tố sau:

* **Training Perplexity:** Khi context dài hơn bậc $N$ tăng, độ bối rối trên tập huấn luyện giảm mạnh. Ngữ cảnh càng dài giúp mô hình ghi nhớ các chuỗi từ cụ thể trong tập huấn luyện tốt hơn, kéo PPL xuống mức rất thấp.
* **Validation và Test Perplexity:** Ngược lại với tập huấn luyện, PPL trên tập validation và test lại tăng vọt khi context dài hơn như kết quả thực nghiệm Trigram tệ hơn rất nhiều so với Bigram và Unigram. Mô hình có context dài dễ bị suy giảm khả năng tổng quát hóa trên dữ liệu mới.
* **Số lượng n-gram:** Khi tăng chiều dài context, không gian biểu diễn của n-gram phình trướng theo cấp số mũ (từ $|V|$ lên $|V|^2$, $|V|^3$,...). Số lượng các tổ hợp từ có thể xảy ra trở nên khổng lồ, vượt quá khả năng lưu trữ và thống kê của corpus.
* **Số lượng unseen n-gram:** Do không gian quá lớn và corpus có giới hạn, tỷ lệ các cụm từ hoàn toàn mới xuất hiện ở tập test bùng nổ mạnh mẽ. Context càng dài thì càng có nhiều n-gram không bao giờ xuất hiện trong tập huấn luyện, khiến mô hình gặp lỗi thiếu hụt tần suất trầm trọng.

**Ý nghĩa chuyển đổi:**
Hiện tượng này chính là điểm nghẽn cốt lõi của phương pháp đếm tần suất truyền thống. Nó dẫn đến sự phát triển tất yếu của lịch sử NLP: chuyển từ **n-gram** sang **neural language model**, tiếp tục tiến hóa qua **word embeddings**, kiến trúc **RNN**, cơ chế **attention**, và đỉnh cao hiện tại là **Transformer** nhằm khắc phục triệt để vấn đề bùng nổ không gian context và dữ liệu thưa.